In [0]:
import pandas as pd
import re
import string

In [0]:
komodo_df = spark.sql("""select distinct npi as hco_npi, ORGANIZATION_NAME as hco_name, PROVIDER_ADDRESS as hco_address, PROVIDER_CITY as hco_city, PROVIDER_STATE as hco_state, PROVIDER_ZIP as hco_zip
from com_raw.kom_providers
where PROVIDER_TYPE = 'ORGANIZATION' and npi in (select distinct hco_npi from cmpa_insights_internal_schema.reference_file_0109)""").toPandas()

In [0]:
%sql
select *
from com_raw.kom_providers
where npi in (select distinct hco_npi from cmpa_insights_internal_schema.reference_file_0109
where hco_name ilike '%atrium%' and hco_npi != '-')

In [0]:
print(komodo_df.shape)
print(komodo_df.columns)
komodo_df.head()

In [0]:
import pandas as pd

data = {
    "Childrens Hospital Los Angeles": "Childrens Hospital Los Angeles ; Children's Hospital Los Angeles ; Childrens Hospital Los Angeles ; CHLA ; Childrens Hosp Los Angeles",
    "Childrens Hospital Of Orange County": "Childrens Hospital Of Orange County ; Children's Hospital of Orange County ; Childrens Hospital Of Orange County ; CHOC ; CHOC Children's",
    "Childrens Hospital Of Philadelphia": "Childrens Hospital Of Philadelphia ; Children's Hospital of Philadelphia ; Childrens Hospital Of Philadelphia ; CHOP ; Childrens Hosp of Philadelphia",
    "Childrens Wisconsin Milwaukee Campus": "Childrens Wisconsin Milwaukee Campus ; Children's Wisconsin ; Childrens Wisconsin ; Children's Hospital of Wisconsin ; CHW",
    "Childrens Mercy Hospital": "Childrens Mercy Hospital ; Children's Mercy Hospital ; Childrens Mercy Hospital ; Children's Mercy ; CMH",
    "Childrens National Hospital": "Childrens National Hospital ; Children's National Hospital ; Childrens National Hospital ; Children's National ; CNH",
    "Childrens Of Alabama": "Childrens Of Alabama ; Children's of Alabama ; Childrens Of Alabama ; COA ; Childrens Alabama",
    "Christus St Vincent Regional Medical Center": "Christus St Vincent Regional Medical Center ; CHRISTUS St. Vincent ; Christus St Vincent ; St Vincent Regional Medical Center ; Christus St. Vincent Regional ; Christus",
    "Christus Childrens San Antonio": "Christus Childrens San Antonio ; CHRISTUS Children's ; Christus Childrens ; Children's Hospital of San Antonio ; Christus Children's Hospital ; Christus",
    "Cincinnati Childrens Hospital Medical Center Burnet Campus": "Cincinnati Childrens Hospital Medical Center Burnet Campus ; Cincinnati Children's Hospital Medical Center ; Cincinnati Childrens Hospital Medical Center ; Cincinnati Children's ; CCHMC",
    "Cook Childrens Medical Center-Fort Worth": "Cook Childrens Medical Center-Fort Worth ; Cook Children's Medical Center ; Cook Childrens Medical Center ; Cook Children's ; Cook Childrens",
    "Driscoll Childrens Hospital-Corpus Christi": "Driscoll Childrens Hospital-Corpus Christi ; Driscoll Children's Hospital ; Driscoll Childrens Hospital ; Driscoll Children's ; Driscoll Childrens ; Driscoll",
    "East Tennessee Childrens Hospital": "East Tennessee Childrens Hospital ; East Tennessee Children's Hospital ; East Tennessee Childrens Hospital ; East Tennessee Children's ; ETCH",
    "Primary Childrens Hospital-Salt Lake City": "Primary Childrens Hospital-Salt Lake City ; Primary Children's Hospital ; Primary Childrens Hospital ; Primary Children's ; PCH",
    "Manning Family Childrens": "Manning Family Childrens ; Manning Family Children's ; Manning Family Childrens ; Manning Children's Hospital ; Manning Family Childrens Hospital ; Childrens Hospital of new Orleans",
    "Tulane University School Of Medicine": "Tulane University School Of Medicine ; Tulane University School of Medicine ; Tulane SOM ; Tulane Medicine ; Tulane Univ School of Medicine",
    "M Health Fairview University Of Minnesota Medical Center-West Bank East Building": "M Health Fairview University Of Minnesota Medical Center-West Bank East Building ; M Health Fairview ; University of Minnesota Medical Center ; UMN Medical Center",
    "MUSC Health Primary Care Lancaster": "MUSC Health Primary Care Lancaster ; MUSC Health ; MUSC Primary Care ; MUSC Health Primary Care ; Medical University of South Carolina",
    "OHSU Hospital Portland": "OHSU Hospital Portland ; OHSU ; OHSU Hospital ; Oregon Health & Science University ; Oregon Health and Science University",
    "Orlando Health Orlando Regional Medical Center": "Orlando Health Orlando Regional Medical Center ; Orlando Regional Medical Center ; Orlando Health ORMC ; ORMC ; Orlando Health",
    "Rady Childrens Hospital San Diego": "Rady Childrens Hospital San Diego ; Rady Children's Hospital ; Rady Childrens Hospital ; Rady Children's ; Rady Childrens ; UCLA Santa Monico",
    "Texas Childrens Hospital": "Texas Childrens Hospital ; Texas Children's Hospital ; Texas Childrens Hospital ; Texas Children's ; TCH",
    "Baylor College Of Medicine-Department Of Pediatrics": "Baylor College Of Medicine-Department Of Pediatrics ; Baylor College of Medicine ; BCM ; Baylor Pediatrics",
    "University Of Kentucky": "University Of Kentucky ; University of Kentucky ; UK HealthCare ; UK",
    "Kentucky Childrens Hospital": "Kentucky Childrens Hospital ; Kentucky Children's Hospital ; Kentucky Childrens Hospital ; Kentucky Children's ; KCH",
    "University Of California At Irvine Health-Orange": "University Of California At Irvine Health-Orange ; UC Irvine Health ; UCI Health ; University of California Irvine Health ; Irvine Health ; Irvine",
    "UC Davis Health Department Of Emergency Medicine": "UC Davis Health Department Of Emergency Medicine ; UC Davis Health ; UCD Health ; Department of Emergency Medicine ; UC Davis Emergency Medicine ; UC Davis",
    "UCLA Health": "UCLA Health ; UCLA Health ; Ronald Reagan UCLA ; UCLA Medical Center ; UCLA",
    "UCSF Benioff Childrens Hospital-Oakland": "UCSF Benioff Childrens Hospital-Oakland ; UCSF Benioff Children's Hospital Oakland ; UCSF Benioff Childrens Hospital Oakland ; Benioff Children's ; Children's Hospital Oakland ; UCSF",
    "UCSD School Of Medicine": "UCSD School Of Medicine ; UC San Diego School of Medicine ; UCSD School of Medicine ; UCSD SOM ; UCSD Medicine ; UCSD",
    "University Of Iowa Healthcare Medical Center": "University Of Iowa Healthcare Medical Center ; University of Iowa Health Care ; UI Health Care ; University of Iowa Hospitals & Clinics ; UIHC",
    "University Of Wisconsin Foundation": "University Of Wisconsin Foundation ; University of Wisconsin Foundation ; UW Foundation ; University of Wisconsin ; UW",
    "Vanderbilt Health Pharmacy At One Hundred Oaks": "Vanderbilt Health Pharmacy At One Hundred Oaks ; Vanderbilt Health Pharmacy ; Vanderbilt Pharmacy ; Vanderbilt",
    "Monroe Carell Jr Childrens Hospital At Vanderbilt": "Monroe Carell Jr Childrens Hospital At Vanderbilt ; Monroe Carell Jr. Children's Hospital ; Monroe Carell Childrens Hospital ; Vanderbilt Children's ; Vanderbilt Childrens",
    "Westchester Medical Center": "Westchester Medical Center ; Westchester Medical Center ; WMC ; WMCHealth ; Westchester Med Ctr",
    "Vanderbilt University Medical Center": "Vanderbilt University Medical Center ; Vanderbilt University Medical Center ; Vanderbilt University",
    "Vanderbilt Medical Group Primary Care": "Vanderbilt Medical Group Primary Care ; Vanderbilt Medical Group",
    "The Greenwood Genetic Center": "The Greenwood Genetic Center ; Greenwood Genetics ; Greenwood Genetics Center",
    "The Permanente Medical Group": "The Permanente Medical Group ; Permanente ; Kaiser Permanente ; Norcal ; Socal ; kaiser",
    "Trustees Of Columbia University In The City Of NY": "Trustees Of Columbia University In The City Of NY ; Columbia university",
    "Atrium Health": "Atrium Health ; Atrium"
}

# Step 1: create dictionary with trimmed keyword lists
keywords_dict = {
    hco: [kw.strip() for kw in keywords.split(";")]
    for hco, keywords in data.items()
}

# Step 2: convert dictionary to DataFrame
keywords_df = pd.DataFrame(
    [(hco, kw_list) for hco, kw_list in keywords_dict.items()],
    columns=["hco_name", "keywords"]
)


In [0]:
keywords_df.display()

In [0]:
def normalize_text(text):
    if pd.isna(text):
        return ""
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [0]:
komodo_df = komodo_df.copy()
komodo_df["hco_name_norm"] = komodo_df["hco_name"].apply(normalize_text)

In [0]:
keyword_rows = []

for _, row in keywords_df.iterrows():
    canonical_hco = row["hco_name"]
    for kw in row["keywords"]:
        keyword_rows.append({
            "canonical_hco": canonical_hco,
            "keyword_raw": kw,
            "keyword_norm": normalize_text(kw)
        })

keywords_lookup_df = pd.DataFrame(keyword_rows)

In [0]:
komodo_df.head()

In [0]:
keywords_lookup_df.head()

In [0]:
def build_pattern(keyword):
    # Word boundaries for multi-word terms
    return rf"\b{re.escape(keyword)}\b"

keywords_lookup_df["pattern"] = keywords_lookup_df["keyword_norm"].apply(build_pattern)


In [0]:
def tag_hco(hco_name_norm, keyword_df):
    matches = []

    for _, row in keyword_df.iterrows():
        if re.search(row["pattern"], hco_name_norm):
            matches.append({
                "canonical_hco": row["canonical_hco"],
                "matched_keyword": row["keyword_raw"],
                "keyword_length": len(row["keyword_norm"])
            })

    if not matches:
        return pd.Series([None, None])

    # Prefer longest keyword (more specific)
    best_match = sorted(matches, key=lambda x: x["keyword_length"], reverse=True)[0]

    return pd.Series([
        best_match["canonical_hco"],
        best_match["matched_keyword"]
    ])


In [0]:
komodo_df[["tagged_hco_name", "matched_keyword"]] = komodo_df["hco_name_norm"].apply(
    lambda x: tag_hco(x, keywords_lookup_df)
)


In [0]:
tagged_df = komodo_df[
    [
        "hco_npi",
        "hco_name",
        "hco_address",
        "hco_city",
        "hco_state",
        "hco_zip",
        "tagged_hco_name",
        "matched_keyword"
    ]
]


In [0]:
tagged_df.display()

In [0]:
%sql
select distinct npi as hco_npi, ORGANIZATION_NAME as hco_name, PROVIDER_ADDRESS as hco_address, PROVIDER_CITY as hco_city, PROVIDER_STATE as hco_state, PROVIDER_ZIP as hco_zip
from com_raw.kom_providers
where PROVIDER_TYPE = 'ORGANIZATION' and npi in (select distinct hco_npi from cmpa_insights_internal_schema.reference_file_0109) and ORGANIZATION_NAME ilike '%greenwood%'


In [0]:
%sql
select * from com_edp_prd.com_raw.vod_hco
where npi_num__v in ('1164686879')

In [0]:
%sql
select * from cmpa_insights_internal_schema.hco360